In [1]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb

df = pd.read_csv(r"C:\Users\joshh\Machine_Learning\Audiology_Project\Audio_Data.csv")
df['INVESTDATE'] = pd.to_datetime(df['INVESTDATE'], format="%d/%m/%Y")

LeftSC = [f'L{f}C' for f in [250, 500, 750, 1000, 1500, 2000, 3000, 4000, 6000, 8000]]
RightSC = [f'R{f}C' for f in [250, 500, 750, 1000, 1500, 2000, 3000, 4000, 6000, 8000]]

df[LeftSC] = df[LeftSC].replace(0, np.nan)
df[RightSC] = df[RightSC].replace(0, np.nan)

df['Right_Mean'] = df[RightSC].mean(axis=1)
df['Left_Mean'] = df[LeftSC].mean(axis=1)

df = df.sort_values(by=['Unique PID', 'INVESTDATE', 'CURVE'])
df['_nan_count'] = df.isnull().sum(axis=1)
df = df.loc[df.groupby(['Unique PID', 'INVESTDATE', 'CURVE'])['_nan_count'].idxmin()].drop(columns=['_nan_count'])

next_df = df.shift(-1)
shift_cols = {col: f'B{col}' for col in LeftSC + RightSC}
shift_cols.update({'Unique PID': 'next_PID', 'INVESTDATE': 'next_DATE', 'CURVE': 'next_CURVE', 'Right_Mean': 'next_Right_Mean', 'Left_Mean': 'next_Left_Mean'})

for old_col, new_col in shift_cols.items():
    df[new_col] = next_df[old_col]

BLeftSC = [f'BL{f}C' for f in [250, 500, 750, 1000, 1500, 2000, 3000, 4000, 6000, 8000]]
BRightSC = [f'BR{f}C' for f in [250, 500, 750, 1000, 1500, 2000, 3000, 4000, 6000, 8000]]


condition = (df['Unique PID'] == df['next_PID']) & (df['INVESTDATE'] == df['next_DATE']) & (df['CURVE'] == 0) & (df['next_CURVE'] == 10)

df['RAC_BC'] = np.where(condition, np.abs(df['Right_Mean'] - df['next_Right_Mean']), np.nan)
df['LAC_BC'] = np.where(condition, np.abs(df['Left_Mean'] - df['next_Left_Mean']), np.nan)

df_end = df.dropna(subset=['RAC_BC', 'LAC_BC'], how='all').copy().reset_index(drop=True)
df_end['Ear_Asymmetry'] = np.abs(df_end['Right_Mean'] - df_end['Left_Mean'])


df_end.drop(columns=['next_PID', 'next_DATE', 'next_CURVE', 'next_Right_Mean', 'next_Left_Mean'], inplace=True)


df_end['BRight_Mean'] = df_end[BRightSC].mean(axis=1)
df_end['BLeft_Mean'] = df_end[BLeftSC].mean(axis=1)

LLeftSC, LRightSC = LeftSC[:5], RightSC[:5]
HLeftSC, HRightSC = LeftSC[5:], RightSC[5:]
LBLeftSC, LBRightSC = BLeftSC[:5], BRightSC[:5]
HBLeftSC, HBRightSC = BLeftSC[5:], BRightSC[5:]

df_end['HBRight_Mean'] = df_end[HRightSC].mean(axis=1)
df_end['HBLeft_Mean'] = df_end[HLeftSC].mean(axis=1)
df_end['HRight_Mean'] = df_end[HBRightSC].mean(axis=1) 
df_end['HLeft_Mean'] = df_end[HBLeftSC].mean(axis=1)   
df_end['LBRight_Mean'] = df_end[LRightSC].mean(axis=1)
df_end['LBLeft_Mean'] = df_end[LLeftSC].mean(axis=1)
df_end['LRight_Mean'] = df_end[LBRightSC].mean(axis=1) 
df_end['LLeft_Mean'] = df_end[LBLeftSC].mean(axis=1)   

df_test = df_end[~((df_end['Treatment Cat'] == 'No Treatment') & ((df_end['Right_Mean'] > 30) | (df_end['Left_Mean'] > 30)))].copy()

engineered_cols = ['Right_Mean', 'Left_Mean', 'RAC_BC', 'LAC_BC', 'Ear_Asymmetry', 'BLeft_Mean', 'BRight_Mean', 'HBRight_Mean', 'HBLeft_Mean', 'HRight_Mean', 'HLeft_Mean', 'LBRight_Mean', 'LBLeft_Mean', 'LRight_Mean', 'LLeft_Mean']
frequency_cols = [c for c in df_test.columns if (c.startswith('R') or c.startswith('L') or c.startswith('B')) and c not in engineered_cols]
feature_cols = frequency_cols + engineered_cols

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
target_names = label_encoder.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=1415, stratify=y_encoded
)

model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=15,
    learning_rate=0.1,
    random_state=1415,
    eval_metric='mlogloss'
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}\n")

print(classification_report(y_test, y_pred, target_names=target_names))

NameError: name 'y' is not defined

In [9]:
import joblib

# Save model and encoder
joblib.dump(model, 'audiology_xgb_model.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')



['label_encoder.pkl']

In [12]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# 1. Load pipeline assets
@st.cache_resource # Keeps the model cached in memory for faster performance
def load_assets():
    model = joblib.load('audiology_xgb_model.pkl')
    encoder = joblib.load('label_encoder.pkl')
    return model, encoder

try:
    model, encoder = load_assets()
except FileNotFoundError:
    st.error("⚠️ Model or Encoder files not found! Please run your training script first.")
    st.stop()

st.title("🦻 Audiology Treatment Prediction Tool")
st.write("Enter the patient's audiogram parameters below to calculate treatment category.")

# Define standard frequencies used in your dataset
frequencies = [250, 500, 750, 1000, 1500, 2000, 3000, 4000, 6000, 8000]

# Create Form Fields
with st.form("audiogram_form"):
    st.subheader("Air Conduction Thresholds (dB HL)")
    col1, col2 = st.columns(2)
    
    left_inputs = {}
    with col1:
        st.markdown("**Left Ear**")
        for f in frequencies:
            left_inputs[f"L{f}C"] = st.number_input(f"L{f}C (dB)", min_value=0, max_value=120, value=20, step=5)
        
    right_inputs = {}
    with col2:
        st.markdown("**Right Ear**")
        for f in frequencies:
            right_inputs[f"R{f}C"] = st.number_input(f"R{f}C (dB)", min_value=0, max_value=120, value=20, step=5)

    submit_button = st.form_submit_button(label="Predict Treatment Class")

if submit_button:
    # Combine user input dictionaries safely
    input_data = {**left_inputs, **right_inputs}
    input_df = pd.DataFrame([input_data])
    
    # 2. Feature Engineering
    # Calculate Mean using speech frequencies (500, 1000, 2000, 4000)
    # 2. Feature Engineering
    # Calculate Mean using speech frequencies (500, 1000, 2000, 4000)
    speech_freqs_r = [f'R{f}C' for f in]
    speech_freqs_l = [f'L{f}C' for f in]

    
    input_df['Right_Mean'] = input_df[speech_freqs_r].mean(axis=1)
    input_df['Left_Mean'] = input_df[speech_freqs_l].mean(axis=1)
    input_df['Ear_Asymmetry'] = np.abs(input_df['Right_Mean'] - input_df['Left_Mean'])
    
    # Ensure column sorting exactly matches what your XGBoost model expects
    feature_cols = [f"L{f}C" for f in frequencies] + [f"R{f}C" for f in frequencies] + ['Right_Mean', 'Left_Mean', 'Ear_Asymmetry']
    
    X_user = input_df[feature_cols]
    
    # 3. Predict class outcome
    pred_encoded = model.predict(X_user)
    pred_name = encoder.inverse_transform(pred_encoded)[0]
    
    st.success(f"🎯 Predicted Treatment Strategy: **{pred_name}**")



SyntaxError: invalid syntax (3919755867.py, line 53)

In [8]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Load pipeline assets
model = joblib.load('audiology_xgb_model.pkl')
encoder = joblib.load('label_encoder.pkl')

st.title("🦻 Audiology Treatment Prediction Tool")
st.write("Enter the patient's audiogram parameters below to calculate treatment category.")

# Create Form Fields
with st.form("audiogram_form"):
    st.subheader("Air Conduction Thresholds (dB HL)")
    col1, col2 = st.columns(2)
    
    with col1:
        st.markdown("**Left Ear**")
        l250 = st.number_input("L250C", min_value=0, max_value=120, value=20)
        l500 = st.number_input("L500C", min_value=0, max_value=120, value=20)
        # Add inputs for all remaining frequencies (750 to 8000)...
        
    with col2:
        st.markdown("**Right Ear**")
        r250 = st.number_input("R250C", min_value=0, max_value=120, value=20)
        r500 = st.number_input("R500C", min_value=0, max_value=120, value=20)
        # Add inputs for all remaining frequencies (750 to 8000)...

    st.subheader("Bone Conduction / Next Curve Data")
    # Add input boxes for BL and BR features if needed, or compute values automatically...
    
    submit_button = st.form_submit_button(label="Predict Treatment Class")

if submit_button:
    # 1. Construct user input dictionary matching your exact raw frequencies
    input_data = {
        'L250C': l250, 'L500C': l500,  # Populate completely
        'R250C': r250, 'R500C': r500,  
    }
    input_df = pd.DataFrame([input_data])
    
    # 2. Replicate your exact feature engineering math inside the app
    input_df['Right_Mean'] = input_df[[f'R{f}C' for f in [250,500]]].mean(axis=1)
    input_df['Left_Mean'] = input_df[[f'L{f}C' for f in [250,500]]].mean(axis=1)
    input_df['Ear_Asymmetry'] = np.abs(input_df['Right_Mean'] - input_df['Left_Mean'])
    # (...Replicate all remaining feature definitions...)

    # Ensure column sorting exactly matches `feature_cols` from your model
    feature_cols = [...] # Define identical list
    X_user = input_df[feature_cols]
    
    # 3. Predict class outcome
    pred_encoded = model.predict(X_user)
    pred_name = encoder.inverse_transform(pred_encoded)[0]
    
    st.success(f"🎯 Predicted Treatment Strategy: **{pred_name}**")

2026-10-01 14:20:22.029 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 14:20:22.029 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 14:20:22.030 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 14:20:22.030 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 14:20:22.030 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 14:20:22.031 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 14:20:22.031 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 14:20:22.032 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar